# GP 候選解釋變數的空間前向選擇

本 notebook 使用真實 TCCIP GRID 的 NN-derived GEV parameter estimates，建立 GP mean structure 的候選變數選擇流程。研究問題是：哪些地形、土地覆蓋與海岸距離變數，能改善未見地理區域的參數預測？


## Block 1：資料與候選變數

三個 response 分別為：

$$
y_\mu(s)=\hat\mu(s),\qquad
y_\sigma(s)=\widehat{\log\sigma}(s),\qquad
y_\xi(s)=\hat\xi(s).
$$


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from spatial_predictor_selection import (
    CANDIDATE_GROUPS,
    calculate_selected_oof_return_levels,
    load_predictor_selection_data,
    predictor_audit,
    run_all_targets,
)

data = load_predictor_selection_data()
print(f'可用 GRID：{len(data):,}')
pd.DataFrame([
    {'candidate_group': group, 'columns': ' + '.join(columns)}
    for group, columns in CANDIDATE_GROUPS.items()
])

## Block 2：候選變數稽核

先檢查缺值、尺度與相關性。相關性只用來辨識可能的共線性，不直接決定變數是否入選；最終選擇仍以 buffered spatial CV 的 OOF RMSE 為準。


In [ ]:
audit, correlations = predictor_audit(data)
display(audit)
display(correlations.head(15))

## Block 3：固定五區與 buffer

先將 GRID 中心投影為 TWD97 / TM2 公里座標，再以 coordinate-based K-means 建立固定的五個 geographic folds。K-means 的作用只有建立空間集中且互斥的 test regions；它不估計空間自相關距離，也不能自行移除 train-test leakage。

$$
\mathcal T_b(r_\delta)=\left\{i:\operatorname{fold}(i)\neq b,\ 
\min_{j\in b}d(s_i,s_j)>r_\delta\right\}.
$$

$$
r_\mu=55\text{ km},\qquad r_{\log\sigma}=35\text{ km},\qquad r_\xi=30\text{ km}.
$$


## Block 4：Spatial Forward Feature Selection

本階段固定前一階段選出的 kernel，只比較 mean structure。從 intercept-only mean 開始，每一步把尚未入選的候選群組逐一加入，所有模型使用完全相同的 folds、buffer 與 training pool。

$$
g^*=\arg\min_{g\in\mathcal G_{\mathrm{remaining}}}
\operatorname{RMSE}_{\mathrm{buffered\ spatial\ CV}}
(\mathcal S_{k-1}\cup g).
$$

$$
\frac{\operatorname{RMSE}_{k-1}-\operatorname{RMSE}_{k}}
{\operatorname{RMSE}_{k-1}}>0.01.
$$


In [ ]:
# 完整 exact-GP FFS 需要數分鐘。若結果 CSV 已存在，預設直接載入。
RUN_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    trials, selection_path, selected_models = run_all_targets(
        data=data,
        n_folds=5,
        max_train=800,
        min_relative_improvement=0.01,
    )
else:
    table_dir = ROOT / 'results' / 'tables'
    trials = pd.read_csv(table_dir / 'spatial_ffs_trials.csv')
    selection_path = pd.read_csv(table_dir / 'spatial_ffs_selection_path.csv')
    selected_models = pd.read_csv(table_dir / 'spatial_ffs_selected_models.csv')

display(selection_path)

## Block 5：目前的開發階段結果

依 1% stopping rule，目前選出的 mean structures 為：

$$
m_\mu(s)=\beta_0+\beta_1\operatorname{Elevation}(s)
+\beta_2\operatorname{LocalRelief}(s)
+\beta_3p_{\mathrm{agriculture}}(s),
$$

$$
m_{\log\sigma}(s)=\beta_0+\beta_1\operatorname{Elevation}(s)
+\beta_2p_{\mathrm{forest}}(s)
+\beta_3\operatorname{CoastDistance}(s),
$$

$$
m_\xi(s)=\beta_0.
$$


In [ ]:
display(selected_models[[
    'target', 'predictors', 'RMSE', 'MAE', 'Bias', 'kernel', 'nu'
]])

# 顯示未通過 1% 門檻的下一個最佳候選。
selected_steps = selected_models.set_index('target')['step'].to_dict()
stopping_rows = []
for target, selected_step in selected_steps.items():
    next_step = trials.query('target == @target and step == @selected_step + 1')
    if not next_step.empty:
        stopping_rows.append(next_step.sort_values('RMSE').iloc[0])
display(pd.DataFrame(stopping_rows)[[
    'target', 'candidate_group', 'relative_RMSE_improvement', 'raw_p'
]])

## Block 6：Selected mixed OOF pipeline 的 return levels

將每個 GRID 的 selected-model OOF location、log-scale 與 shape predictions 一對一合併，建立 mixed OOF pipeline：

$$
RL_T(s)=\mu(s)+\frac{\sigma(s)}{\xi(s)}
\left[\{-\log(1-1/T)\}^{-\xi(s)}-1\right].
$$


In [ ]:
table_dir = ROOT / 'results' / 'tables'
selected_oof = pd.read_csv(
    table_dir / 'spatial_ffs_selected_oof_predictions.csv'
)
rl_metrics, rl_fold_metrics, rl_oof_predictions = (
    calculate_selected_oof_return_levels(
        selected_oof,
        return_periods=(50, 100),
        output_directory=table_dir,
    )
)
display(rl_metrics.round(4))
display(rl_fold_metrics.round(4))


### Block 6 結果判讀

- $RL_{50}$ pooled OOF RMSE 為 1.2906，MAE 為 0.9220，Bias 為 -0.0516。
- $RL_{100}$ pooled OOF RMSE 為 1.3599，MAE 為 0.9552，Bias 為 -0.0964。
- 兩個 return periods 的 finite rate 都是 1.0，沒有無限或數值發散的 return levels。
- Fold 1 的 RMSE 分別為 2.3198 與 2.4231，明顯高於其他區域；因此整體平均不能掩蓋 geographic fold instability。
